# 4. Build the Agent Loop Yourself

In the last notebook, Python sent one prompt to the OpenAI API and read one answer.

An **agent loop** adds one more idea: the model can ask your Python code to run a tool. Your code runs the tool, sends the result back, and lets the model continue.

The loop is still made from regular API requests:

- send a prompt and a list of tools
- receive a tool call from the model
- run the matching Python function
- send the tool result back
- repeat until the model gives a final answer


## Today's Goals

- Define a Python function that can act as a tool.
- Describe that function with a JSON schema.
- Ask the model a question that needs the tool.
- Inspect the model's `function_call`.
- Send a `function_call_output` back to the model.
- Put the steps into a reusable agent loop.


In [ ]:
import json
import os
from pprint import pprint

import requests
# If you do not have python-dotenv installed, run: pip install python-dotenv
from dotenv import load_dotenv

TIMEOUT = 60
MODEL = "gpt-4.1-mini"


## Load the API Key

This is the same setup from notebook 3. The key lives in `.env`, not inside the notebook.


In [ ]:
load_dotenv()

api_key = os.environ["OPENAI_API_KEY"]

print("OPENAI_API_KEY loaded")


## Prepare the OpenAI Request

We will keep using the Responses API directly with `requests`.


In [ ]:
url = "https://api.openai.com/v1/responses"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

print("Endpoint:", url)
print("Method: POST")


## A Normal Python Function

This function uses the public Open-Meteo APIs from notebook 2.

By itself, it is just Python code. Soon we will describe it to the model as a tool the model can ask us to run.


In [ ]:
geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"
forecast_url = "https://api.open-meteo.com/v1/forecast"


def get_current_weather(city_name):
    geocoding_params = {"name": city_name, "count": 1}
    geocoding_response = requests.get(
        geocoding_url,
        params=geocoding_params,
        timeout=TIMEOUT,
    )
    geocoding_response.raise_for_status()

    location_data = geocoding_response.json()
    if "results" not in location_data:
        return {"error": f"No location found for {city_name}"}

    location = location_data["results"][0]

    forecast_params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "current": "temperature_2m,wind_speed_10m",
    }
    forecast_response = requests.get(
        forecast_url,
        params=forecast_params,
        timeout=TIMEOUT,
    )
    forecast_response.raise_for_status()

    forecast = forecast_response.json()
    current = forecast["current"]
    units = forecast["current_units"]

    return {
        "city": location["name"],
        "country": location.get("country"),
        "temperature": current["temperature_2m"],
        "temperature_unit": units["temperature_2m"],
        "wind_speed": current["wind_speed_10m"],
        "wind_speed_unit": units["wind_speed_10m"],
    }


In [ ]:
pprint(get_current_weather("Tokyo"))


## Describe the Function as a Tool

The model cannot run Python by itself. It only sees the tool description we send in the API request.

The `parameters` field is a JSON schema. It tells the model what arguments the function expects.


In [ ]:
tools = [
    {
        "type": "function",
        "name": "get_current_weather",
        "description": (
            "Get the current temperature and wind speed for a city using Open-Meteo. "
            "Use this when the user asks about current weather."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "city_name": {
                    "type": "string",
                    "description": "The city to look up, like Tokyo, Paris, or San Francisco.",
                }
            },
            "required": ["city_name"],
            "additionalProperties": False,
        },
        "strict": True,
    }
]

pprint(tools)


## Ask the Model to Use the Tool

For this first run, we will require a tool call so the notebook reliably shows the tool-calling shape.

The important thing to notice: the model does not return the weather yet. It returns a request for Python to call `get_current_weather`.


In [ ]:
payload = {
    "model": MODEL,
    "input": "What is the current weather in Tokyo? Answer in one sentence.",
    "tools": tools,
    "tool_choice": "required",
    "parallel_tool_calls": False,
    "max_output_tokens": 300,
}

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=TIMEOUT,
)

print(response.status_code)
response.raise_for_status()


In [ ]:
response_data = response.json()

print("Response id:", response_data["id"])
print("Output item types:", [item["type"] for item in response_data["output"]])
pprint(response_data["output"])


## Read the Function Call

A function call has three pieces we care about:

- `name`: which tool the model wants
- `arguments`: a JSON string with the function inputs
- `call_id`: an ID we must include when we send the tool result back


In [ ]:
function_calls = [
    item for item in response_data["output"]
    if item["type"] == "function_call"
]

function_call = function_calls[0]

print("Tool name:", function_call["name"])
print("Call id:", function_call["call_id"])
print("Raw arguments:", function_call["arguments"])

arguments = json.loads(function_call["arguments"])
pprint(arguments)


## Run the Tool in Python

Now Python does the real work.

The model chose the tool and arguments, but our code decides what function is allowed to run. That routing step is an important safety habit.


In [ ]:
def call_tool(function_call):
    tool_name = function_call["name"]
    arguments = json.loads(function_call["arguments"])

    if tool_name == "get_current_weather":
        return get_current_weather(**arguments)

    raise ValueError(f"Unknown tool: {tool_name}")


tool_result = call_tool(function_call)
pprint(tool_result)


## Send the Tool Result Back

The model asked for a tool call. Python ran the tool.

Now we send back a `function_call_output` item with the same `call_id`, so the model knows which tool call this result belongs to.


In [ ]:
follow_up_payload = {
    "model": MODEL,
    "previous_response_id": response_data["id"],
    "input": [
        {
            "type": "function_call_output",
            "call_id": function_call["call_id"],
            "output": json.dumps(tool_result),
        }
    ],
    "tools": tools,
    "parallel_tool_calls": False,
    "max_output_tokens": 300,
}

follow_up_response = requests.post(
    url,
    headers=headers,
    json=follow_up_payload,
    timeout=TIMEOUT,
)

print(follow_up_response.status_code)
follow_up_response.raise_for_status()


## Read the Final Answer

The final response should contain normal output text because the model now has the tool result.


In [ ]:
def extract_response_text(response_data):
    pieces = []

    for item in response_data["output"]:
        for content in item.get("content", []):
            if content.get("type") == "output_text":
                pieces.append(content["text"])

    return "\n".join(pieces)


final_data = follow_up_response.json()

print("Output item types:", [item["type"] for item in final_data["output"]])
print(extract_response_text(final_data))


## Turn Those Steps Into a Loop

An agent loop repeats the same pattern until there are no more function calls.

This version uses `previous_response_id` so the API can connect each turn to the previous one. The code still owns the loop.


In [ ]:
def create_response(payload):
    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=TIMEOUT,
    )
    response.raise_for_status()
    return response.json()


def extract_function_calls(response_data):
    return [
        item for item in response_data["output"]
        if item["type"] == "function_call"
    ]


def run_agent(user_question, max_turns=5):
    payload = {
        "model": MODEL,
        "instructions": (
            "You are a concise assistant. Use the available tools when they help "
            "answer with current facts."
        ),
        "input": user_question,
        "tools": tools,
        "parallel_tool_calls": False,
        "max_output_tokens": 300,
    }

    response_data = create_response(payload)

    for turn_number in range(1, max_turns + 1):
        output_types = [item["type"] for item in response_data["output"]]
        print(f"Turn {turn_number}: {output_types}")

        function_calls = extract_function_calls(response_data)
        if not function_calls:
            return extract_response_text(response_data)

        tool_outputs = []
        for function_call in function_calls:
            print(f"Calling {function_call['name']} with {function_call['arguments']}")
            tool_result = call_tool(function_call)

            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": function_call["call_id"],
                    "output": json.dumps(tool_result),
                }
            )

        response_data = create_response(
            {
                "model": MODEL,
                "previous_response_id": response_data["id"],
                "input": tool_outputs,
                "tools": tools,
                "parallel_tool_calls": False,
                "max_output_tokens": 300,
            }
        )

    raise RuntimeError("Agent stopped because it reached max_turns")


In [ ]:
answer = run_agent(
    "What is the current weather in Paris? If the wind is above 20 km/h, tell me to bring a jacket."
)

print(answer)


## What You Built

The model did not directly call the weather API. Your Python code did.

That is the main idea behind an agent loop:

- the model decides what it needs next
- your code checks and runs the requested tool
- your code returns the result
- the model uses the result to continue

In real projects, you would add more checks: validate arguments, handle API errors, limit the number of turns, log tool calls, and ask a human before running risky actions.


## Quick Review

- A **tool** is a function your application makes available to the model.
- A **function call** is the model asking your code to run a tool.
- A **function call output** is your code sending the tool result back.
- An **agent loop** repeats model calls and tool calls until the model has enough information to answer.
- The model chooses, but your code stays in control of what actually runs.
